# NutriPlate YOLOv8 Multi-Stage Training (Low Cam + High Cam)

Notebook ini dibuat untuk Kaggle dan langsung jalan copy-paste.
Strategi: train 3 tahap berurutan dari `yolov8s.pt` + augmentasi low-camera (blur/noise/compression/low light).


In [ ]:
!pip install -q ultralytics albumentations opencv-python pyyaml

In [ ]:
from pathlib import Path
import random
import shutil
import zipfile
import cv2
import yaml
import numpy as np
import albumentations as A
from tqdm import tqdm
from ultralytics import YOLO

# =========================
# KONFIGURASI UTAMA
# =========================
SEED = 42
FAST_MODE = True  # set False for full training
np.random.seed(SEED)
random.seed(SEED)

SOURCE_DATA_DIR = Path('/kaggle/input/nutriplate-merged-target-35/data_merged')  # ganti sesuai dataset Kaggle
SOURCE_YAML = SOURCE_DATA_DIR / 'data.yaml'

WORK_DATA_DIR = Path('/kaggle/working/data_merged_lowcam')
WORK_YAML = Path('/kaggle/working/data_lowcam.yaml')

PROJECT_DIR = Path('/kaggle/working/NutriPlate/train')
RUN_NAME_PREFIX = 'nutriplate_y8s'

# VRAM Anda 32GB: mulai batch besar, fallback otomatis jika OOM
BATCH_CANDIDATES = [16, 12, 8]

# Intensitas augmentasi low-camera
AUG_PROB = 0.25         # probabilitas gambar train dibuat versi low-cam
AUG_PER_IMAGE = 1       # jumlah salinan low-cam per gambar

assert SOURCE_DATA_DIR.exists(), f'Dataset tidak ditemukan: {SOURCE_DATA_DIR}'
assert SOURCE_YAML.exists(), f'data.yaml tidak ditemukan: {SOURCE_YAML}'
print('Dataset source:', SOURCE_DATA_DIR)
IMG_SIZE = 1024
BATCH_CANDIDATES = [16, 12, 8]
START_WEIGHTS = Path('/kaggle/input/nutriplate-models/1.0.pt')
if not START_WEIGHTS.exists():
    START_WEIGHTS = Path('yolov8s.pt')
print('Start weights:', START_WEIGHTS)

In [ ]:
# =========================
# COPY DATASET KE /kaggle/working
# =========================
if WORK_DATA_DIR.exists():
    shutil.rmtree(WORK_DATA_DIR)

shutil.copytree(SOURCE_DATA_DIR, WORK_DATA_DIR)
print('Dataset copied to:', WORK_DATA_DIR)


In [ ]:
# =========================
# AUGMENTASI LOW-CAMERA (TRAIN ONLY)
# Original train images tetap ada -> robust high-cam
# Tambahan low-cam images -> robust low-cam
# =========================
train_img_dir = WORK_DATA_DIR / 'images' / 'train'
train_lbl_dir = WORK_DATA_DIR / 'labels' / 'train'

img_ext = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
image_paths = sorted([p for p in train_img_dir.iterdir() if p.suffix.lower() in img_ext])

# Catatan: transform di bawah adalah photometric-only (tanpa ubah geometri),
# jadi bbox tetap sama; kita tidak pakai bbox_params agar tidak gagal karena
# floating-point out-of-range dari label sumber.
transform = A.Compose(
    [
        A.OneOf([
            A.MotionBlur(blur_limit=(3, 9), p=1.0),
            A.GaussianBlur(blur_limit=(3, 7), p=1.0),
            A.Defocus(radius=(2, 5), alias_blur=(0.1, 0.4), p=1.0),
        ], p=0.55),
        A.OneOf([
            A.GaussNoise(std_range=(0.02, 0.08), mean_range=(0.0, 0.0), p=1.0),
            A.ISONoise(color_shift=(0.01, 0.04), intensity=(0.1, 0.35), p=1.0),
        ], p=0.55),
        A.OneOf([
            A.ImageCompression(quality_range=(28, 70), p=1.0),
            A.Downscale(
                scale_range=(0.50, 0.80),
                interpolation_pair={'downscale': cv2.INTER_AREA, 'upscale': cv2.INTER_LINEAR},
                p=1.0,
            ),
        ], p=0.55),
        A.RandomBrightnessContrast(brightness_limit=0.30, contrast_limit=0.25, p=0.65),
        A.HueSaturationValue(hue_shift_limit=6, sat_shift_limit=18, val_shift_limit=20, p=0.30),
        A.RandomShadow(shadow_roi=(0.0, 0.4, 1.0, 1.0), num_shadows_limit=(1, 2), p=0.25),
    ]
)

def sanitize_yolo_bbox(x, y, w, h, eps=1e-6):
    # Konversi ke xyxy lalu clip, agar pasti valid di [0,1]
    x1 = max(0.0, min(1.0, x - w / 2.0))
    y1 = max(0.0, min(1.0, y - h / 2.0))
    x2 = max(0.0, min(1.0, x + w / 2.0))
    y2 = max(0.0, min(1.0, y + h / 2.0))

    nw = x2 - x1
    nh = y2 - y1
    if nw <= eps or nh <= eps:
        return None

    nx = (x1 + x2) / 2.0
    ny = (y1 + y2) / 2.0

    # Clamp akhir untuk hindari 1.00000x karena floating point
    nx = min(1.0 - eps, max(eps, nx))
    ny = min(1.0 - eps, max(eps, ny))
    nw = min(1.0 - 2 * eps, max(eps, nw))
    nh = min(1.0 - 2 * eps, max(eps, nh))

    return [nx, ny, nw, nh]

def read_yolo_label(path: Path):
    if not path.exists():
        return [], []

    bboxes, cls_ids = [], []
    for line in path.read_text().splitlines():
        parts = line.split()
        if len(parts) != 5:
            continue

        c, x, y, w, h = parts
        try:
            cls_id = int(float(c))
            x = float(x)
            y = float(y)
            w = float(w)
            h = float(h)
        except ValueError:
            continue

        fixed = sanitize_yolo_bbox(x, y, w, h)
        if fixed is None:
            continue

        cls_ids.append(cls_id)
        bboxes.append(fixed)

    return bboxes, cls_ids

def write_yolo_label(path: Path, bboxes, cls_ids):
    lines = []
    for c, (x, y, w, h) in zip(cls_ids, bboxes):
        lines.append(f"{int(c)} {x:.6f} {y:.6f} {w:.6f} {h:.6f}")
    path.write_text('\n'.join(lines))

added = 0
skipped_invalid = 0
for img_path in tqdm(image_paths, desc='Generating low-cam variants'):
    if random.random() > AUG_PROB:
        continue

    img = cv2.imread(str(img_path))
    if img is None:
        continue

    lbl_path = train_lbl_dir / f'{img_path.stem}.txt'
    bboxes, cls_ids = read_yolo_label(lbl_path)

    # Kalau label asli ada tapi setelah sanitize kosong, skip augment image ini
    if lbl_path.exists() and not bboxes:
        skipped_invalid += 1
        continue

    for i in range(AUG_PER_IMAGE):
        transformed = transform(image=img)
        out_img = transformed['image']

        new_stem = f'{img_path.stem}__lowcam_{i}'
        out_img_path = train_img_dir / f'{new_stem}.jpg'
        out_lbl_path = train_lbl_dir / f'{new_stem}.txt'

        cv2.imwrite(str(out_img_path), out_img)
        write_yolo_label(out_lbl_path, bboxes, cls_ids)
        added += 1

print(f'Low-cam augmented images added: {added}')
print(f'Skipped due to invalid bbox after sanitize: {skipped_invalid}')
print(f'Train images total now: {len(list(train_img_dir.glob("*")))}')


In [ ]:
# =========================
# NORMALIZER: Downsample train by class cap
# Jalankan ini jika dataset terlalu bias (contoh: tempe terlalu banyak)
# =========================
from collections import defaultdict

MAX_PER_CLASS = 2000  # ubah sesuai kebutuhan
SEED = 42

rng = random.Random(SEED)

train_img_dir = WORK_DATA_DIR / 'images' / 'train'
train_lbl_dir = WORK_DATA_DIR / 'labels' / 'train'

img_ext = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
train_images = [p for p in train_img_dir.iterdir() if p.suffix.lower() in img_ext]

# kumpulkan image per class
per_class = defaultdict(list)
for img_path in train_images:
    lbl_path = train_lbl_dir / f'{img_path.stem}.txt'
    if not lbl_path.exists():
        continue
    for line in lbl_path.read_text().splitlines():
        parts = line.split()
        if len(parts) < 5:
            continue
        try:
            cid = int(float(parts[0]))
        except ValueError:
            continue
        per_class[cid].append(img_path)

# ambil subset agar tidak melebihi MAX_PER_CLASS
keep = set()
for cid, imgs in per_class.items():
    if len(imgs) <= MAX_PER_CLASS:
        keep.update(imgs)
    else:
        keep.update(rng.sample(imgs, MAX_PER_CLASS))

# hapus image/label yang tidak terpilih
removed = 0
for img_path in train_images:
    if img_path not in keep:
        lbl_path = train_lbl_dir / f'{img_path.stem}.txt'
        if lbl_path.exists():
            lbl_path.unlink()
        img_path.unlink()
        removed += 1

print(f'Downsample selesai. Removed {removed} train images')
print(f'Remaining train images: {len(keep)}')


In [ ]:
# =========================
# CLEANUP LABEL TRAIN (SATUKAN FORMAT & CLAMP BBOX)
# Penting untuk stabilkan box regression
# =========================
train_lbl_dir = WORK_DATA_DIR / 'labels' / 'train'

fixed_count = 0
removed_lines = 0

def sanitize_line(parts, eps=1e-6):
    if len(parts) != 5:
        return None
    try:
        c = int(float(parts[0]))
        x, y, w, h = map(float, parts[1:])
    except ValueError:
        return None

    x1 = max(0.0, min(1.0, x - w / 2.0))
    y1 = max(0.0, min(1.0, y - h / 2.0))
    x2 = max(0.0, min(1.0, x + w / 2.0))
    y2 = max(0.0, min(1.0, y + h / 2.0))

    nw = x2 - x1
    nh = y2 - y1
    if nw <= eps or nh <= eps:
        return None

    nx = min(1.0 - eps, max(eps, (x1 + x2) / 2.0))
    ny = min(1.0 - eps, max(eps, (y1 + y2) / 2.0))
    nw = min(1.0 - 2 * eps, max(eps, nw))
    nh = min(1.0 - 2 * eps, max(eps, nh))
    return f"{c} {nx:.6f} {ny:.6f} {nw:.6f} {nh:.6f}"

for lbl_path in train_lbl_dir.glob('*.txt'):
    old_lines = lbl_path.read_text().splitlines()
    new_lines = []
    for line in old_lines:
        fixed = sanitize_line(line.split())
        if fixed is None:
            removed_lines += 1
            continue
        new_lines.append(fixed)

    if ''.join(old_lines) != ''.join(new_lines):
        fixed_count += 1
        lbl_path.write_text(''.join(new_lines))

print(f'Label files cleaned: {fixed_count}')
print(f'Invalid lines removed: {removed_lines}')


In [ ]:
# =========================
# BUILD TRAIN YAML DARI DATA WORKING
# =========================
source_cfg = yaml.safe_load(SOURCE_YAML.read_text())
names = source_cfg['names']

cfg = {
    'path': str(WORK_DATA_DIR),
    'train': 'images/train',
    'val': 'images/valid',
    'test': 'images/test',
    'nc': len(names),
    'names': names,
}

WORK_YAML.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True))
print('Train YAML:', WORK_YAML)
print(WORK_YAML.read_text())

In [ ]:
# =========================
# HELPER TRAINING (AUTO FALLBACK BATCH SAAT OOM)
# =========================
def train_stage(weights, name, epochs, lr0, lrf, freeze, mosaic, mixup, close_mosaic, patience, use_multiscale=False):
    last_err = None

    for bsz in BATCH_CANDIDATES:
        print(f"\n[TRAIN] {name} | weights={weights} | batch={bsz} | multi_scale={use_multiscale}")
        try:
            model = YOLO(weights)
            model.train(
                data=str(WORK_YAML),
                epochs=epochs,
                imgsz=1024,
                batch=bsz,
                optimizer='AdamW',
                lr0=lr0,
                lrf=lrf,
                box=7.5,
                dfl=1.6 if FAST_MODE else 2.0,
                iou=0.60,
                freeze=freeze,
                patience=patience,
                device=0,
                workers=8,
                cache='disk',
                amp=True,
                cos_lr=True,
                pretrained=True,
                seed=42,
                deterministic=True,
                # Generalization all camera types
                mosaic=mosaic,
                mixup=mixup,
                copy_paste=0.02,
                erasing=0.12,
                close_mosaic=close_mosaic,
                degrees=4.0,
                translate=0.10,
                scale=0.40,
                shear=1.0,
                perspective=0.0005,
                hsv_h=0.02,
                hsv_s=0.80,
                hsv_v=0.50,
                fliplr=0.5,
                flipud=0.01,
                multi_scale=use_multiscale,
                project=str(PROJECT_DIR),
                name=name,
                exist_ok=True,
            )
            return PROJECT_DIR / name / 'weights' / 'best.pt'
        except ZeroDivisionError as e:
            last_err = e
            if use_multiscale:
                print('ZeroDivisionError terdeteksi pada multi_scale, retry dengan multi_scale=False ...')
                use_multiscale = False
                continue
            raise
        except RuntimeError as e:
            last_err = e
            msg = str(e).lower()
            if 'out of memory' in msg or 'cuda' in msg:
                print(f'OOM/Device issue on batch={bsz}, coba batch lebih kecil...')
                continue
            raise

    raise RuntimeError(f'Gagal training stage {name} di semua batch candidates') from last_err


In [ ]:
# =========================
# CLEANUP TRAINING FOLDER (WAJIB kalau checkpoint corrupt)
# =========================
import shutil
shutil.rmtree(PROJECT_DIR, ignore_errors=True)


In [ ]:
# =========================
# STAGE 1: warm-up dari yolov8s.pt
# =========================
stage1_best = train_stage(
    weights=str(START_WEIGHTS),
    name=f'{RUN_NAME_PREFIX}_stage1_warm',
    epochs=120,
    lr0=0.0008,
    lrf=0.010,
    freeze=10,
    mosaic=0.40,
    mixup=0.0,
    close_mosaic=12,
    patience=12 if FAST_MODE else 20,
    use_multiscale=False,
)
print('Stage 1 best:', stage1_best)
assert stage1_best.exists(), f'Model stage1 tidak ditemukan: {stage1_best}'

In [ ]:
# =========================
# STAGE 2: refine pakai best stage1
# =========================
stage2_best = train_stage(
    weights=str(stage1_best),
    name=f'{RUN_NAME_PREFIX}_stage2_refine',
    epochs=160,
    lr0=0.0012,
    lrf=0.004,
    freeze=4,
    mosaic=0.15,
    mixup=0.0,
    close_mosaic=8,
    patience=15 if FAST_MODE else 25,
    use_multiscale=False,
)
print('Stage 2 best:', stage2_best)
assert stage2_best.exists(), f'Model stage2 tidak ditemukan: {stage2_best}'

In [ ]:
# =========================
# STAGE 3: final fine-tune pakai best stage2
# =========================
stage3_best = train_stage(
    weights=str(stage2_best),
    name=f'{RUN_NAME_PREFIX}_stage3_final',
    epochs=200,
    lr0=0.0006,
    lrf=0.0010,
    freeze=0,
    mosaic=0.00,
    mixup=0.0,
    close_mosaic=0,
    patience=18 if FAST_MODE else 30,
    use_multiscale=False,
)
print('Stage 3 best:', stage3_best)
assert stage3_best.exists(), f'Model stage3 tidak ditemukan: {stage3_best}'

In [ ]:
# =========================
# EVALUASI FINAL (TEST SPLIT)
# =========================
best_model = YOLO(str(stage3_best))
metrics = best_model.val(data=str(WORK_YAML), split='test', imgsz=IMG_SIZE, device=0)
print(metrics)
print('Final best model:', stage3_best)

In [ ]:
# =========================
# ZIP HASIL TRAINING
# =========================
zip_path = Path('/kaggle/working/nutriplate_results.zip')
if zip_path.exists():
    zip_path.unlink()

root_to_zip = Path('/kaggle/working/NutriPlate')
if root_to_zip.exists():
    shutil.make_archive('/kaggle/working/nutriplate_results', 'zip', str(root_to_zip))
    print('Zipping selesai:', zip_path)
else:
    print('Folder training tidak ditemukan:', root_to_zip)